In [1]:
%matplotlib inline
import random
import torch
from d2l import torch as d2l

In [2]:
class SyntheticRegressionData(d2l.DataModule):
  def __init__(self, w, b, noise=0.01, num_train=1000, num_val=1000, batch_size=32, drop_last=False):
    super().__init__()
    self.save_hyperparameters()
    n = num_train + num_val
    
    self.X = torch.randn(n, len(w))
    noise = torch.randn(n, 1) * noise
    
    self.y = torch.matmul(self.X, w.reshape((-1, 1))) + b + noise

In [3]:
data = SyntheticRegressionData(w=torch.tensor([2, -3.4]), b=4.2)

In [4]:
print('features:', data.X[0],'\nlabel:', data.y[0])

features: tensor([-0.1983, -0.3088]) 
label: tensor([4.8766])


In [5]:
@d2l.add_to_class(SyntheticRegressionData)
def get_dataloader(self, train):
    if train:
        indices = list(range(0, self.num_train))
        # The examples are read in random order
        random.shuffle(indices)
    else:
        indices = list(range(self.num_train, self.num_train+self.num_val))
    for i in range(0, len(indices), self.batch_size):
        batch_indices = torch.tensor(indices[i: i+self.batch_size])
        yield self.X[batch_indices], self.y[batch_indices]

In [14]:
@d2l.add_to_class(d2l.DataModule)
def get_tensor_loader(self, tensors, train, indices=slice(0, None), drop_last=False):
  tensors = tuple(a[indices] for a in tensors)
  dataset = torch.utils.data.TensorDataset(*tensors)
  return torch.utils.data.DataLoader(dataset, self.batch_size, shuffle=train, drop_last=drop_last)

@d2l.add_to_class(SyntheticRegressionData)
def get_dataloader(self, train, drop_last=False):
  i = slice(0, self.num_train) if train else slice(self.num_train, None)
  return self.get_tensorloader((self.X, self.y), train, i, drop_last=drop_last)

In [15]:
data = SyntheticRegressionData(w=torch.tensor([2, -3.4]), b=4.2, num_train=100, num_val=100)

In [16]:
for X_batch, y_batch in data.train_dataloader():
  print(f"X_batch: {X_batch.shape}")
  print(f"y_batch: {y_batch.shape}")

TypeError: DataModule.get_tensorloader() got an unexpected keyword argument 'drop_last'

In [ ]:
data = SyntheticRegressionData(w=torch.tensor([2, -3.4]), b=4.2, num_train=100, num_val=100, drop_last=True)
